# Query Common Crawl in milliseconds

The standard cost model for analyzing a Common Crawl sample from Python is:
**every question re-parses every page**. Whether you use BeautifulSoup or lxml,
the parsed trees die with the process, so tomorrow's question pays the full
parse bill again.

[htmlarc](https://github.com/akesson/htmlarc) changes the model: parse the
corpus **once** into a `.htmlarc` archive of ready-to-query DOMs, then every
question after that is a CSS selector sweep over a memory-mapped file — no
HTML parsing at read time, all cores, GIL released.

This notebook does the whole thing live, and races lxml (in its
best-practice fast configuration) on the same data:

1. Stream ~1,000 pages of the latest Common Crawl into RAM
2. Build the archive (the one-time cost, measured)
3. Ask the corpus questions in milliseconds
4. Ask lxml the same questions (it has to re-parse — measured)
5. Ask a question invented *after* the "crawl" — the actual point

**Requirements**: `pip install htmlarc polars lxml cssselect warcio requests`
— until htmlarc lands on PyPI, build the wheel locally instead (two commands,
see [the recipes README](README.md#running)).

In [1]:
import os
import time

import htmlarc
import polars as pl

LIMIT = int(os.environ.get("CC_SAMPLE_DOCS", "1000"))  # pages to stream
ARCHIVE = "data/cc_notebook.htmlarc"


def timed(label, fn):
    """Run fn(), print the wall time, return (result, seconds)."""
    t0 = time.perf_counter()
    out = fn()
    dt = time.perf_counter() - t0
    print(f"{dt * 1e3:9.1f} ms  {label}")
    return out, dt

## 1. Stream a sample of the latest crawl

Common Crawl publishes each crawl as WARC files on plain HTTPS. We stream the
first `text/html` responses out of the crawl's first WARC file — no
full-segment download — and decode them to `str`, which is the input all the
parsers here get. (The retry loop matters: CC's frontend readily answers
`503 Slow Down` under load.)

In [2]:
import gzip

import requests
from warcio.archiveiterator import ArchiveIterator

CC = "https://data.commoncrawl.org"
UA = {"User-Agent": "htmlarc-notebook/0.1 (+https://github.com/akesson/htmlarc)"}


def get(url, tries=9, **kw):
    """GET with retry/backoff for Common Crawl's 503/502-under-load habit."""
    kw.setdefault("timeout", 60)
    for attempt in range(tries):
        try:
            r = requests.get(url, headers=UA, **kw)
        except (requests.ConnectionError, requests.Timeout):
            if attempt == tries - 1:
                raise
            time.sleep(min(2 ** (attempt + 1), 60))
            continue
        if r.status_code in (503, 429, 502, 504) and attempt < tries - 1:
            time.sleep(min(2 ** (attempt + 1), 60))
            continue
        r.raise_for_status()
        return r


def decode(body, content_type):
    """bytes -> str, honoring a declared charset, falling back to UTF-8."""
    if content_type and "charset=" in content_type:
        cs = content_type.split("charset=")[-1].split(";")[0].strip().strip("\"'")
        try:
            return body.decode(cs, errors="replace")
        except LookupError:
            pass
    return body.decode("utf-8", errors="replace")


crawl = get("https://index.commoncrawl.org/collinfo.json").json()[0]["id"]
warc = gzip.decompress(get(f"{CC}/crawl-data/{crawl}/warc.paths.gz").content).decode().splitlines()[0]

pages = []  # (key, html_str, meta) — held in RAM so every contender gets identical input
t0 = time.perf_counter()
with get(f"{CC}/{warc}", stream=True) as r:
    for rec in ArchiveIterator(r.raw):
        if rec.rec_type != "response" or rec.http_headers is None:
            continue
        ct = rec.http_headers.get_header("Content-Type")
        if not ct or "text/html" not in ct or rec.http_headers.get_statuscode() != "200":
            continue
        body = rec.content_stream().read()
        if not body:
            continue
        url = rec.rec_headers.get_header("WARC-Target-URI")
        pages.append((f"{url}#{len(pages)}", decode(body, ct), {"url": url, "bytes": len(body)}))
        if len(pages) >= LIMIT:
            break

html_mb = sum(m["bytes"] for _, _, m in pages) / 1e6
print(f"{len(pages)} pages, {html_mb:.0f} MB of HTML from {crawl}, "
      f"streamed in {time.perf_counter() - t0:.0f} s")

1000 pages, 68 MB of HTML from CC-MAIN-2026-30, streamed in 6 s


## 2. Build the archive — the one-time cost

One pass: parse every page (HTML5-style recovery, so wild web markup doesn't
crash anything) and write the DOMs plus **typed per-document metadata
columns** into a single file. This is the only time any HTML gets parsed in
this notebook.

In [3]:
def build():
    with htmlarc.ArchiveBuilder(ARCHIVE, on_error="skip",
                                meta_schema={"url": str, "bytes": int}) as b:
        for key, html, meta in pages:
            b.add(key, html, meta=meta)


os.makedirs("data", exist_ok=True)
_, build_s = timed(f"parse {len(pages)} pages once -> {ARCHIVE}", build)

arc_mb = os.path.getsize(ARCHIVE) / 1e6
print(f"archive: {arc_mb:.0f} MB ({arc_mb / html_mb:.2f}x the source HTML — "
      "a pre-parsed DOM store, with text zstd-compressed per bundle)")

    752.5 ms  parse 1000 pages once -> data/cc_notebook.htmlarc
archive: 51 MB (0.75x the source HTML — a pre-parsed DOM store, with text zstd-compressed per bundle)


## 3. Ask the corpus questions

Opening the archive is a memory-map — effectively free. The sweeps run the
compiled CSS selector over every document's stored DOM in Rust, on all
cores, with the GIL released.

In [4]:
arc, open_s = timed("open the archive (mmap)", lambda: htmlarc.open(ARCHIVE))

# Q1 — where does this slice of the web link? scan_table returns the whole
# sweep as one Arrow table: a row per matched element, and the typed metadata
# columns ride along on every row (no sidecar file, no join).
links, q1_s = timed("Q1  scan_table('a[href]') -> Arrow -> polars",
                    lambda: pl.DataFrame(arc.scan_table("a[href]", attrs=["href"], meta=["url"])))
top = (links.with_columns(domain=pl.col("href").str.extract(r"^https?://([^/]+)", 1))
       .drop_nulls("domain")
       .group_by("domain")
       .agg(links=pl.len(), pages=pl.col("url").n_unique())
       .sort("links", descending=True)
       .head(8))
print(f"{len(links)} links total; top external domains:")
top

      0.4 ms  open the archive (mmap)
     25.6 ms  Q1  scan_table('a[href]') -> Arrow -> polars
109734 links total; top external domains:


domain,links,pages
str,u32,u32
"""tuparejaal100.com""",2114,1
"""jaderland.nl""",753,1
"""melanielinktaylor.mzteachuh.or…",741,1
"""artharbour-ao.blogspot.jp""",737,1
"""video.sina.com.cn""",691,2
"""www.3tui.net""",681,1
"""artharbour-ao.blogspot.com""",653,1
"""sukucadangalatberat.id""",488,1


In [5]:
# Q2 — when the question is "how many", nothing needs to cross into Python at
# all: scan_count counts inside Rust and returns one int.
n_links, c1_s = timed("Q2a scan_count('a[href]')", lambda: arc.scan_count("a[href]", attr="href"))
n_heads, c2_s = timed("Q2b scan_count('h1, h2, h3')", lambda: arc.scan_count("h1, h2, h3"))
print(f"{n_links} links and {n_heads} headings across {len(arc)} pages")

      1.6 ms  Q2a scan_count('a[href]')
      1.7 ms  Q2b scan_count('h1, h2, h3')
109734 links and 6095 headings across 1000 pages


## 4. The race: lxml on the same data

lxml is the fast incumbent, and it gets its best-practice configuration
here: pre-compiled `CSSSelector`s, bytes input (its native mode), and for
the counting question an XPath `count()` evaluated entirely inside libxml2 —
no per-match Python objects. What it cannot avoid is **re-parsing every page
per question**, because that's the workflow: parsed trees don't survive the
session, and a new question means a new pass.

In [6]:
import lxml.html
from lxml import etree
from lxml.cssselect import CSSSelector

sel_links = CSSSelector("a[href]")
count_links = etree.XPath(f"count({sel_links.path})")
count_heads = etree.XPath(f"count({CSSSelector('h1, h2, h3').path})")


def lxml_pass(per_tree):
    acc = 0
    for _key, html, _meta in pages:
        try:
            tree = lxml.html.fromstring(html.encode())  # bytes: lxml's native input
        except lxml.etree.ParserError:
            continue  # empty/degenerate documents raise
        acc += per_tree(tree)
    return acc


lx_links, lx1_s = timed("lxml Q1  re-parse + extract hrefs",
                        lambda: lxml_pass(lambda t: len([a.get("href") for a in sel_links(t)])))
lx_count, lx2_s = timed("lxml Q2  re-parse + XPath count()",
                        lambda: lxml_pass(lambda t: int(count_links(t)) + int(count_heads(t))))
print(f"lxml sees {lx_links} links (htmlarc: {n_links} — small tree-recovery/charset deltas are normal)")

    650.2 ms  lxml Q1  re-parse + extract hrefs


    574.7 ms  lxml Q2  re-parse + XPath count()
lxml sees 103824 links (htmlarc: 109734 — small tree-recovery/charset deltas are normal)


In [7]:
rows = [
    ("Q1: every href (+ source url)", lx1_s, q1_s),
    ("Q2: count links + headings", lx2_s, c1_s + c2_s),
]
pl.DataFrame({
    "question": [r[0] for r in rows],
    "lxml (re-parse, 1 core)": [f"{r[1] * 1e3:,.0f} ms" for r in rows],
    "htmlarc (archive, all cores)": [f"{r[2] * 1e3:,.0f} ms" for r in rows],
    "speedup": [f"{r[1] / r[2]:,.0f}x" for r in rows],
})

question,"lxml (re-parse, 1 core)","htmlarc (archive, all cores)",speedup
str,str,str,str
"""Q1: every href (+ source url)""","""650 ms""","""26 ms""","""25x"""
"""Q2: count links + headings""","""575 ms""","""3 ms""","""176x"""


Two honest notes on that table:

- **The first pass is not where htmlarc wins.** The archive build above cost
  about as much as one lxml pass — parse-speed-wise the two are within
  ~1.2–1.3× of each other. If you will only ever look at a corpus once,
  lxml is entirely competitive. The table is about every pass *after* the
  first.
- **Cores**: the lxml column is one core; the htmlarc sweeps use all of
  them (that's the point — the GIL is released, parallelism is free).
  lxml users scale with `multiprocessing`, re-parsing in every worker; the
  [full benchmark](../../benchmarks/python-compare/README.md) measures that
  too, and core-for-core htmlarc still counts ~25× faster and sweeps ~4–5×
  faster on a 5,000-doc / 405 MB segment. Holding all lxml trees in RAM
  instead costs ~8× the HTML size in memory and *still* queries slower than
  the mmap'd archive.

## 5. The question you think of tomorrow

The extraction schema of an `.htmlarc` archive is not fixed at crawl time —
the archive stores the *DOMs*, so any future CSS selector is fair game.
Here's a question invented long after the "crawl": *how many pages ship
JSON-LD structured data, and how many link out over plain http?*

In [8]:
jsonld = 'script[type="application/ld+json"]'
n_ld, t_ld = timed("pages with JSON-LD", lambda: len(arc.matching(jsonld)))
n_http, t_http = timed("scan_count(a[href^='http://'])",
                       lambda: arc.scan_count("a[href^='http://']"))
print(f"{n_ld}/{len(arc)} pages carry JSON-LD; {n_http} insecure http:// links")

# lxml answering the same new question = yet another full re-parse of the corpus:
sel_ld = CSSSelector(jsonld)
_, lx3_s = timed("lxml, same question (re-parse again)",
                 lambda: lxml_pass(lambda t: bool(sel_ld(t))))
print(f"\nlxml: {lx3_s:.2f} s per new question, forever. "
      f"htmlarc: {(t_ld + t_http) * 1e3:.0f} ms ({lx3_s / (t_ld + t_http):,.0f}x) — "
      "and the archive is a file you can ship to a teammate.")

      2.8 ms  pages with JSON-LD
      4.0 ms  scan_count(a[href^='http://'])
170/1000 pages carry JSON-LD; 52017 insecure http:// links


    558.9 ms  lxml, same question (re-parse again)

lxml: 0.56 s per new question, forever. htmlarc: 7 ms (81x) — and the archive is a file you can ship to a teammate.


## What this looks like at real scale

This notebook's sample is deliberately small. On the
[full measured benchmark](../../benchmarks/python-compare/README.md)
(5,000 Common Crawl docs, 405 MB, Apple M4 Pro), per question over the corpus:

| | time | vs htmlarc sweep |
|---|---|---|
| BeautifulSoup (re-parse, 1 core) | 32.8 s | ~330× slower |
| lxml (re-parse, 1 core) | 3.04 s | ~30× slower |
| lxml (re-parse, `multiprocessing`, 14 cores) | 0.52 s | ~5× slower |
| **htmlarc `scan_*`** (all cores) | **0.10 s** | — |
| **htmlarc `scan_count`** (counting only) | **0.019 s** | 25× vs 14-core lxml count |

And that still spots the incumbents their input pre-decoded in RAM. Starting
from the compressed `.warc.gz` on disk — the bigger-than-memory reality —
*just reading and decoding the source* costs 7× more than htmlarc needs to
answer the entire question (0.73 s vs 0.10 s), a floor no amount of
parallelism takes lxml below. The archive itself came out **0.69× the size
of the source HTML** on this corpus.

Caveats, so you can trust the rest: the htmlarc Python API is read/query-only
(no DOM mutation), input is pre-decoded `str` (no charset detection), the
selector language is CSS3 (no XPath), and one-shot extraction is only
~1.2–1.3× faster than lxml — the order-of-magnitude wins are specifically in
the parse-once-ask-many workflow you just ran.

**Next steps**: the [runnable recipes](README.md) cover appending to an
archive in place, snapshot diffing, RAG chunking, and ZIM ingestion;
[`quickstart.py`](quickstart.py) is the 5-minute API tour.